# Rolling Style Drift

<p> The puporse of this notebook is too run rolling 36 months OLS for each ETF

<li>
    <ul> Checking on the beta over time </ul>
    <ul> Detect any style drift in each ETF </ul>
    <ul> Save Rolling.csv file </ul>
</li>

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import statsmodels.api as sm 
from pathlib import Path

In [4]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(PROCESSED_DIR / "etf_ff5_with_predictions.csv")
df["date"] = pd.to_datetime(df["date"])

df.head()

,date,ticker,return,mkt_rf,smb,hml,rmw,cma,rf,excess_return,predicted_excess_return,predicted_return,residual,cumulative_actual_return,cumulative_predicted_return,cumulative_residual
0,2015-02-28,IWM,0.059464,0.0614,0.0036,-0.0179,-0.0110,-0.0175,0.0,0.059464,0.063394,0.063394,-0.003930,0.059464,0.063394,-0.003930
1,2015-03-31,IWM,0.017700,-0.0109,0.0308,-0.0038,0.0007,-0.0062,0.0,0.017700,0.012998,0.012998,0.004702,0.078216,0.077216,0.000753
2,2015-04-30,IWM,-0.025649,0.0060,-0.0301,0.0180,0.0005,-0.0062,0.0,-0.025649,-0.017461,-0.017461,-0.008188,0.050561,0.058407,-0.007441
3,2015-05-31,IWM,0.022364,0.0138,0.0082,-0.0111,-0.0176,-0.0083,0.0,0.022364,0.020681,0.020681,0.001682,0.074055,0.080296,-0.005771
4,2015-06-30,IWM,0.007829,-0.0154,0.0290,-0.0082,0.0035,-0.0154,0.0,0.007829,0.006559,0.006559,0.001271,0.082464,0.087381,-0.004508


In [5]:
factor_cols = ["mkt_rf", "smb", "hml", "rmw", "cma"]
window = 36

rolling_rows = []

for ticker, group in df.groupby("ticker"):
    group = group.sort_values("date").reset_index(drop=True)

    for end_idx in range(window, len(group) + 1):
        window_data = group.iloc[end_idx - window:end_idx].copy()

        y = window_data["excess_return"]
        X = sm.add_constant(window_data[factor_cols])

        model = sm.OLS(y, X).fit()

        end_date = window_data["date"].iloc[-1]

        rolling_rows.append({
            "date": end_date,
            "ticker": ticker,
            "rolling_alpha": model.params["const"],
            "rolling_beta_market": model.params["mkt_rf"],
            "rolling_beta_smb": model.params["smb"],
            "rolling_beta_hml": model.params["hml"],
            "rolling_beta_rmw": model.params["rmw"],
            "rolling_beta_cma": model.params["cma"],
            "rolling_r_squared": model.rsquared,
            "window_months": window,
            "n_obs": int(model.nobs),
        })

rolling_betas = pd.DataFrame(rolling_rows)

rolling_betas.head()

,date,ticker,rolling_alpha,rolling_beta_market,rolling_beta_smb,rolling_beta_hml,rolling_beta_rmw,rolling_beta_cma,rolling_r_squared,window_months,n_obs
0,2018-01-31,IWM,-0.001267,0.995340,0.808727,0.130364,0.012806,-0.191150,0.988630,36,36
1,2018-02-28,IWM,-0.001455,1.015335,0.799303,0.123295,-0.001284,-0.163228,0.988047,36,36
2,2018-03-31,IWM,-0.001297,1.009866,0.805331,0.117774,-0.005677,-0.156728,0.987428,36,36
3,2018-04-30,IWM,-0.001029,1.009946,0.784373,0.139035,-0.016479,-0.168780,0.988971,36,36
4,2018-05-31,IWM,-0.001134,1.009891,0.782940,0.142404,-0.009284,-0.169141,0.989607,36,36


In [6]:
rolling_betas.shape

(882, 11)

In [7]:
rolling_betas["ticker"].value_counts()

ticker
IWM     98
MTUM    98
QQQ     98
QUAL    98
SPY     98
USMV    98
VLUE    98
VTV     98
VUG     98
Name: count, dtype: int64

In [8]:
rolling_betas[rolling_betas["ticker"] == "QQQ"].tail()

,date,ticker,rolling_alpha,rolling_beta_market,rolling_beta_smb,rolling_beta_hml,rolling_beta_rmw,rolling_beta_cma,rolling_r_squared,window_months,n_obs
289,2025-10-31,QQQ,0.003052,1.066043,-0.027325,-0.466123,0.011898,-0.134557,0.929637,36,36
290,2025-11-30,QQQ,0.002917,1.044969,-0.022542,-0.470828,-0.035093,-0.146126,0.931477,36,36
291,2025-12-31,QQQ,0.005231,0.988096,0.017536,-0.530437,-0.078327,-0.033950,0.926308,36,36
292,2026-01-31,QQQ,0.005708,0.982982,0.020482,-0.522937,-0.060884,-0.005242,0.916365,36,36
293,2026-02-28,QQQ,0.004915,1.000381,0.001309,-0.505017,-0.062687,-0.021927,0.919215,36,36


In [9]:
rolling_betas.to_csv(
    PROCESSED_DIR / "rolling_factor_betas.csv",
    index=False
)